## 3.05 Boolean Expressions HW

[Source lesson and criteria](https://pages.opencodingsociety.com/python/boolean/2027)

Completed exercises, predictions, runnable solutions, and recorded execution results follow. Each Python cell includes its own setup so it can run independently in the published runner.


### Popcorn Hack 1 — Check SFI Part Fields (0.10)

Prediction: all five field/category checks are True for the supplied flywheel record. Removing the product name makes only the name check False. The category check accepts Auto Racing or Drag Racing.


In [1]:
# CODE_RUNNER: Popcorn Hack 1 - Check SFI Part Fields
def field_checks(part):
    return {
        "product_name": bool(part.get("product_name", "").strip()),
        "category": part.get("category") in ["Auto Racing", "Drag Racing"],
        "spec_number": bool(part.get("spec_number", "").strip()),
        "effective_date": bool(part.get("effective_date", "").strip()),
    }

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing", "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001",
}
print("Supplied record:", field_checks(part))
assert all(field_checks(part).values())
changed = dict(part, product_name="")
print("Missing name:", field_checks(changed))
assert not field_checks(changed)["product_name"]
print("Both field-check tests passed.")


Supplied record: {'product_name': True, 'category': True, 'spec_number': True, 'effective_date': True}
Missing name: {'product_name': False, 'category': True, 'spec_number': True, 'effective_date': True}
Both field-check tests passed.


### Recorded execution result

```text
Supplied record: {'product_name': True, 'category': True, 'spec_number': True, 'effective_date': True}
Missing name: {'product_name': False, 'category': True, 'spec_number': True, 'effective_date': True}
Both field-check tests passed.
```


### Popcorn Hack 2 — Backend Create Rule (0.15)

Prediction: the original spec 1.1 is rejected because it already exists. Changing it to 1.2 permits creation. `and` requires every field rule; `not` rejects duplicate specs.


In [2]:
# CODE_RUNNER: Popcorn Hack 2 - Backend Create Rule
part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing", "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001",
}
valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]
def can_create(candidate):
    required_valid = (bool(candidate["product_name"].strip())
                      and candidate["category"] in valid_categories
                      and bool(candidate["spec_number"].strip())
                      and bool(candidate["effective_date"].strip()))
    duplicate_spec = candidate["spec_number"] in existing_spec_numbers
    return required_valid and not duplicate_spec

print("Original duplicate:", can_create(part))
print("Changed unique spec:", can_create(dict(part, spec_number="1.2")))
assert can_create(part) is False
assert can_create(dict(part, spec_number="1.2")) is True


Original duplicate: False
Changed unique spec: True


### Recorded execution result

```text
Original duplicate: False
Changed unique spec: True
```


### Popcorn Hack 3 — SFI Search Filter (0.20)

Prediction: query 1.2 returns Multiple Disc Clutch Assemblies only. Query flywheel returns both flywheel records. A Street Car match must be excluded even when its name matches.


In [3]:
# CODE_RUNNER: Popcorn Hack 3 - SFI Search Filter
records = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]
valid_categories = ["Auto Racing", "Drag Racing"]
def search(records, query):
    matches = []
    for record in records:
        name_matches = query.lower() in record["product_name"].lower()
        spec_matches = query == record["spec_number"]
        if record["category"] in valid_categories and (name_matches or spec_matches):
            matches.append(record)
    return matches

for query in ["1.2", "flywheel", "missing"]:
    print(query, "=>", [r["spec_number"] for r in search(records, query)])
assert [r["spec_number"] for r in search(records, "1.2")] == ["1.2"]
assert len(search(records, "flywheel")) == 2
invalid = {"product_name": "Flywheel", "category": "Street Car", "spec_number": "3.1"}
assert search([invalid], "Flywheel") == []
assert search(records, "missing") == []


1.2 => ['1.2']
flywheel => ['1.1', '2.1']
missing => []


### Recorded execution result

```text
1.2 => ['1.2']
flywheel => ['1.1', '2.1']
missing => []
```


### MCQ (0.15)

MCQ 3.05: 4/4 | answers: A,C,A,B

1. A — `7 >= 7` includes equality and evaluates to True.
2. C — both name and spec are required, so use `and`.
3. A — `not False` is True.
4. B — either category comparison may be true, so use `or`.

These written answers are checked against the lesson's answer key.


### Homework Hack — SFI Backend Validator (0.40)

Prediction: supplied spec 1.2 is accepted, Street Car spec 3.1 is rejected, and duplicate spec 1.1 is rejected. The reusable validator returns a Boolean plus field-specific reasons so a backend can explain rejected records.


In [4]:
# CODE_RUNNER: Homework - SFI Backend Validator
existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]
test_records = [
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.2", "effective_date": "Feb. 9, 2006"},
    {"product_name": "Replacement Flywheels", "category": "Street Car", "spec_number": "3.1", "effective_date": "Jan. 1, 2026"},
    {"product_name": "Existing Flywheel Record", "category": "Drag Racing", "spec_number": "1.1", "effective_date": "Jan. 1, 2026"},
]

def validate_record(candidate, existing_specs):
    """Return (accepted: bool, reasons: list) for required fields and unique spec."""
    reasons = []
    for field in ["product_name", "spec_number", "effective_date"]:
        value = candidate.get(field, "")
        if not isinstance(value, str) or not value.strip():
            reasons.append(field + " is required")
    if candidate.get("category") not in valid_categories:
        reasons.append("category is not accepted")
    if candidate.get("spec_number") in existing_specs:
        reasons.append("spec number already exists")
    return not reasons, reasons

decisions = []
for candidate in test_records:
    accepted, reasons = validate_record(candidate, existing_spec_numbers)
    decisions.append(accepted)
    print(candidate["spec_number"], "ACCEPTED" if accepted else "REJECTED",
          "; ".join(reasons))
assert decisions == [True, False, False]

# Exercise every required-field rejection separately, including whitespace.
valid = test_records[0]
for field in ["product_name", "spec_number", "effective_date"]:
    invalid = dict(valid, **{field: "   "})
    assert validate_record(invalid, existing_spec_numbers)[0] is False
assert validate_record({}, existing_spec_numbers)[0] is False
assert validate_record(dict(valid, category="Drag Racing"), existing_spec_numbers)[0]
print("Required fields, both accepted categories, invalid category and duplicates passed.")


1.2 ACCEPTED 
3.1 REJECTED category is not accepted
1.1 REJECTED spec number already exists
Required fields, both accepted categories, invalid category and duplicates passed.


### Recorded execution result

```text
1.2 ACCEPTED 
3.1 REJECTED category is not accepted
1.1 REJECTED spec number already exists
Required fields, both accepted categories, invalid category and duplicates passed.
```


### Explanation and rubric evidence

All required rules are combined: field presence AND an accepted category AND NOT a duplicate. Search uses (name match OR exact spec match) AND an accepted category. In College Board pseudocode these map to AND, OR, NOT, and `←` for assignment; Python uses `=` for assignment and `==` for equality.

The three separate Popcorn runners expose each intermediate Boolean or search result. The homework includes the supplied three records plus empty and whitespace-field cases. Published notebook: [Boolean homework](https://myhomies123.github.io/portfolio/python/boolean-hw). Submit the Homework cell's Python code and this link through the lesson's Code Submission form.


MCQ 3.05: 4/4 | answers: A,C,A,B

Written answers checked against the lesson's answer key: all four match. This is the checked written-answer result; the lesson's interactive quiz/form is a separate submission step.
